In [56]:
import pandas as pd

In [57]:
experiments = [
    'baseline', 'lukasiewicz', 'min', 'l1', 'entropy', 'sign_ste', 'sign_sigmoid', 
    'temp_linear', 'temp_logarithmic', 'temp_geometric', 'temp_inversed', 'temp_back_inversed'
]
domains = ['MIS', 'MaxCut']
dregs = [3, 5]
graph_sizes = [100]

In [58]:
# ['experiment', 'domain', 'd_regular', 'graph_size'] + ['qubo_loss', 'pred_size', 'solver_size', 'violation', 'improvement']
result_df = pd.DataFrame(columns=['graph_id', 'rnd_seed', 'experiment', 'domain', 'd_regular', 'graph_size', 'qubo_loss', 'pred_size', 'solver_size', 'violation', 'improvement'])
for exp in experiments:
    for dom in domains:
        for dreg in dregs:
            for gsize in graph_sizes:
                for rnd_seed in range(10):
                    curr_df = pd.read_csv(f'perf_results/{exp}/{dom}/{dreg}/{gsize}/{rnd_seed}.csv')
                    curr_df = curr_df.assign(rnd_seed=rnd_seed, experiment=exp, domain=dom, d_regular=dreg, graph_size=gsize)
                    curr_df = curr_df.reset_index(names='graph_id')
                    result_df = pd.concat([result_df if not result_df.empty else None, curr_df])
display(result_df.head())

,graph_id,qubo_loss,pred_size,solver_size,violation,improvement,rnd_seed,experiment,domain,d_regular,graph_size
0,0,-41.974800,42,38,0,4,0,baseline,MIS,3,100
1,1,-39.976002,40,35,0,5,0,baseline,MIS,3,100
2,2,-42.974201,43,37,0,6,0,baseline,MIS,3,100
3,3,-40.975399,41,37,0,4,0,baseline,MIS,3,100
4,4,-39.975998,40,40,0,0,0,baseline,MIS,3,100


In [59]:
asdf = result_df[result_df['domain'] == 'MaxCut']
display(asdf[['graph_id', 'qubo_loss', 'pred_size', 'solver_size', 'violation', 'improvement']])
print(asdf[asdf['improvement'] < 0])

,graph_id,qubo_loss,pred_size,solver_size,violation,improvement
0,0,-125.949600,126,100,0,26
1,1,-119.952003,120,100,0,20
2,2,-123.948792,128,100,0,28
3,3,-118.951599,121,100,0,21
4,4,-119.951996,120,100,0,20
...,...,...,...,...,...,...
15,15,-174.926025,185,100,0,85
16,16,-176.927612,181,100,0,81
17,17,-168.929230,177,100,0,77
18,18,-173.927216,182,100,0,82


    graph_id  qubo_loss  pred_size  solver_size  violation  improvement  \
19        19   0.000000          0          100          0         -100   
0          0   0.000000          0          100          0         -100   
1          1   0.000000          0          100          0         -100   
2          2   0.000000          0          100          0         -100   
3          3   0.000000          0          100          0         -100   
..       ...        ...        ...          ...        ...          ...   
15        15   0.000005          0          100          0         -100   
16        16   0.000005          0          100          0         -100   
19        19   0.000005          0          100          0         -100   
7          7   0.000005          0          100          0         -100   
5          5 -43.971199         72          100          0          -28   

    rnd_seed    experiment  domain  d_regular  graph_size  
19         0   lukasiewicz  MaxCut     

In [60]:
def get_best_of_seeds(df, metric, is_max=True):
    df = df[df['violation'] == 0]
    group = df.groupby(['experiment', 'graph_id'])[metric]
    if is_max:
        result = group.max()
    else:
        result = group.min()
    return result

def get_avg_of_seeds(df, metric, penalize=True):
    if penalize:
        df.loc[df['violation'] != 0, metric] = 0
    else:
        df = df[df['violation'] == 0]
    group = df.groupby(['experiment', 'graph_id'])[metric]
    mean, std = group.mean(), group.std()
    return mean, std

def find_best_per_graph(best_df, metric, is_max=True):
    best_records = []
    for i in range(20):
        filtered = best_df.xs(i, level=1)
        best_value = filtered.max() if is_max else filtered.min()
        best_filtered = filtered[filtered == best_value]
        for index, value in best_filtered.items():
            best_records.append([i, index, value])
        # best_index = filtered.idxmax() if is_max else filtered.idxmin()
        # best_record = filtered.loc[best_index]    
        # best_records.append([i, best_index, best_record])
    best_per_graph_df = pd.DataFrame(best_records, columns=['graph_id', 'experiment', metric])
    return best_per_graph_df

def how_many_in_best(best_per_graph_df):
    res_dict = []
    for exp in experiments:
        filtered = best_per_graph_df[best_per_graph_df['experiment'] == exp]
        res_dict.append((exp, len(filtered.index)))
    res_df = pd.DataFrame(res_dict, columns=['experiment', 'best_count'])
    return res_df

def find_best_average(df, metric, is_max=True, penalize=True):
    if penalize:
        df.loc[df['violation'] != 0, metric] = 0
    else:
        df = df[df['violation'] == 0]
    mean = df.groupby('experiment')[metric].mean()
    best_index = mean.idxmax() if is_max else mean.idxmin()
    best_record = mean.loc[best_index]
    return best_index, best_record

def sort_by_average(df, metric, ascending=False, penalize=True):
    if penalize:
        df.loc[df['violation'] != 0, metric] = 0
    else:
        df = df[df['violation'] == 0]
    mean = df.groupby('experiment')[metric].mean()
    sorted_df = mean.sort_values(ascending=ascending)
    return sorted_df

## 1. MIS, d=3, gsize=100

In [61]:
df1 = result_df[(result_df['domain'] == 'MIS') & (result_df['d_regular'] == 3) & (result_df['graph_size'] == 100)]

In [62]:
best_df1 = get_best_of_seeds(df1, 'pred_size', is_max=True)
best_per_graph_df1 = find_best_per_graph(best_df1, 'pred_size', is_max=True)
# display(best_per_graph_df1)
display(how_many_in_best(best_per_graph_df1))

,experiment,best_count
0,baseline,16
1,lukasiewicz,2
2,min,7
3,l1,15
4,entropy,16
5,sign_ste,0
6,sign_sigmoid,1
7,temp_linear,13
8,temp_logarithmic,16
9,temp_geometric,16


In [63]:
avg_df1, _ = get_avg_of_seeds(df1, 'pred_size')
best_per_graph_avg_df1 = find_best_per_graph(avg_df1, 'pred_size', is_max=True)
# display(best_per_graph_avg_df1)
display(how_many_in_best(best_per_graph_avg_df1))

,experiment,best_count
0,baseline,4
1,lukasiewicz,0
2,min,1
3,l1,1
4,entropy,4
5,sign_ste,0
6,sign_sigmoid,0
7,temp_linear,0
8,temp_logarithmic,2
9,temp_geometric,2


In [64]:
best_idx1, best_rec1 = find_best_average(df1, 'pred_size', is_max=True)
print(best_idx1, best_rec1)

baseline 41.195


In [65]:
sorted_df1 = sort_by_average(df1, 'pred_size')
display(sorted_df1)

experiment
baseline              41.195
entropy               41.195
temp_back_inversed    41.195
temp_geometric        41.085
temp_logarithmic      41.085
temp_inversed         41.060
min                   40.630
l1                    40.545
temp_linear           40.465
sign_sigmoid          31.885
lukasiewicz           28.905
sign_ste              22.735
Name: pred_size, dtype: float64

## 2. MaxCut, d=3, gsize=100

In [66]:
df2 = result_df[(result_df['domain'] == 'MaxCut') & (result_df['d_regular'] == 3) & (result_df['graph_size'] == 100)]

In [67]:
best_df2 = get_best_of_seeds(df2, 'pred_size', is_max=True)
best_per_graph_df2 = find_best_per_graph(best_df2, 'pred_size', is_max=True)
# display(best_per_graph_df2)
display(how_many_in_best(best_per_graph_df2))

,experiment,best_count
0,baseline,12
1,lukasiewicz,1
2,min,4
3,l1,12
4,entropy,12
5,sign_ste,0
6,sign_sigmoid,0
7,temp_linear,8
8,temp_logarithmic,9
9,temp_geometric,9


In [68]:
avg_df2, _ = get_avg_of_seeds(df2, 'pred_size')
best_per_graph_avg_df2 = find_best_per_graph(avg_df2, 'pred_size', is_max=True)
# display(best_per_graph_avg_df2)
display(how_many_in_best(best_per_graph_avg_df2))

,experiment,best_count
0,baseline,3
1,lukasiewicz,0
2,min,0
3,l1,1
4,entropy,3
5,sign_ste,0
6,sign_sigmoid,0
7,temp_linear,0
8,temp_logarithmic,4
9,temp_geometric,4


In [69]:
best_idx2, best_rec2 = find_best_average(df2, 'pred_size', is_max=True)
print(best_idx2, best_rec2)

temp_inversed 125.76


In [70]:
sorted_df2 = sort_by_average(df2, 'pred_size')
display(sorted_df2)

experiment
temp_inversed         125.760
temp_back_inversed    125.355
baseline              125.340
entropy               125.340
temp_geometric        125.090
temp_logarithmic      125.090
l1                    125.060
temp_linear           124.620
min                   121.175
lukasiewicz           120.450
sign_sigmoid          107.775
sign_ste               93.915
Name: pred_size, dtype: float64

## 3. MIS, d=5, gsize=100

In [71]:
df3 = result_df[(result_df['domain'] == 'MIS') & (result_df['d_regular'] == 5) & (result_df['graph_size'] == 100)]

In [72]:
best_df3 = get_best_of_seeds(df3, 'pred_size', is_max=True)
best_per_graph_df3 = find_best_per_graph(best_df3, 'pred_size', is_max=True)
# display(best_per_graph_df3)
display(how_many_in_best(best_per_graph_df3))

,experiment,best_count
0,baseline,15
1,lukasiewicz,3
2,min,7
3,l1,2
4,entropy,15
5,sign_ste,0
6,sign_sigmoid,0
7,temp_linear,6
8,temp_logarithmic,11
9,temp_geometric,11


In [73]:
avg_df3, _ = get_avg_of_seeds(df3, 'pred_size')
best_per_graph_avg_df3 = find_best_per_graph(avg_df3, 'pred_size', is_max=True)
# display(best_per_graph_avg_df3)
display(how_many_in_best(best_per_graph_avg_df3))

,experiment,best_count
0,baseline,6
1,lukasiewicz,0
2,min,4
3,l1,0
4,entropy,6
5,sign_ste,0
6,sign_sigmoid,0
7,temp_linear,1
8,temp_logarithmic,3
9,temp_geometric,3


In [74]:
best_idx3, best_rec3 = find_best_average(df3, 'pred_size', is_max=True)
print(best_idx3, best_rec3)

min 33.26


In [75]:
sorted_df3 = sort_by_average(df3, 'pred_size')
display(sorted_df3)

experiment
min                   33.260
temp_geometric        32.900
temp_logarithmic      32.900
baseline              32.825
entropy               32.825
temp_linear           32.640
temp_back_inversed    32.375
temp_inversed         31.760
lukasiewicz           25.125
sign_sigmoid          23.475
sign_ste              18.855
l1                     5.385
Name: pred_size, dtype: float64

## 4.MaxCut, d=5, gsize=100

In [76]:
df4 = result_df[(result_df['domain'] == 'MaxCut') & (result_df['d_regular'] == 5) & (result_df['graph_size'] == 100)]

In [77]:
best_df4 = get_best_of_seeds(df4, 'pred_size', is_max=True)
best_per_graph_df4 = find_best_per_graph(best_df4, 'pred_size', is_max=True)
display(best_per_graph_df4)
display(how_many_in_best(best_per_graph_df4))

,graph_id,experiment,pred_size
0,0,min,199
1,1,min,202
2,2,min,196
3,3,min,199
4,4,temp_linear,196
5,5,min,193
6,6,min,201
7,7,min,197
8,8,min,203
9,9,min,195


,experiment,best_count
0,baseline,1
1,lukasiewicz,1
2,min,15
3,l1,2
4,entropy,1
5,sign_ste,0
6,sign_sigmoid,0
7,temp_linear,3
8,temp_logarithmic,2
9,temp_geometric,2


In [78]:
avg_df4, _ = get_avg_of_seeds(df4, 'pred_size')
best_per_graph_avg_df4 = find_best_per_graph(avg_df4, 'pred_size', is_max=True)
display(best_per_graph_avg_df4)
display(how_many_in_best(best_per_graph_avg_df4))

,graph_id,experiment,pred_size
0,0,min,192.9
1,1,min,192.6
2,2,min,190.1
3,3,min,191.6
4,4,min,190.1
5,5,min,188.0
6,6,min,193.4
7,7,min,192.2
8,8,baseline,193.2
9,8,entropy,193.2


,experiment,best_count
0,baseline,1
1,lukasiewicz,0
2,min,18
3,l1,0
4,entropy,1
5,sign_ste,0
6,sign_sigmoid,0
7,temp_linear,0
8,temp_logarithmic,0
9,temp_geometric,0


In [79]:
best_idx4, best_rec4 = find_best_average(df4, 'pred_size', is_max=True)
print(best_idx4, best_rec4)

min 191.045


In [80]:
sorted_df4 = sort_by_average(df4, 'pred_size')
display(sorted_df4)

experiment
min                   191.045
temp_inversed         185.555
baseline              184.160
entropy               184.160
temp_back_inversed    184.120
temp_geometric        183.375
temp_logarithmic      183.375
l1                    183.240
temp_linear           182.120
lukasiewicz           153.515
sign_sigmoid          139.375
sign_ste              127.650
Name: pred_size, dtype: float64